# E04: Dataclasses y Tipado Estatico

## Objetivos de aprendizaje

Al finalizar esta sesion, seras capaz de:

1. Crear clases orientadas a datos con `@dataclass` y eliminar el boilerplate de las clases manuales.
2. Controlar el comportamiento avanzado: `default_factory`, `frozen=True`, `slots=True`, `order=True`, `kw_only` y `field()`.
3. Validar y calcular atributos derivados con `__post_init__`.
4. Inspeccionar y transformar dataclasses con `replace`, `asdict` y `fields`.
5. Escribir anotaciones de tipo modernas (`X | None`, `Generic[T]`, `Literal`, `Protocol`) y entender los beneficios del type checking estatico con `mypy`.

## Analogia: Formularios y planos (blueprints)

### Dataclasses = formularios que se auto-completan

Imagina llenar a mano la misma tabla de Excel una y otra vez: nombre, email, salario, fecha de alta... cada vez escribes los encabezado, formulas y formato desde cero. **Dataclasses** es la plantilla que ya trae todo eso hecho: tu solo nombras los campos y Python escribe por ti los metodos `__init__`, `__repr__` y `__eq__`. Es como un formulario pre-diseñado que se auto-completa.

### typing = plano de construccion (blueprint)

En una obra de construccion, el **plano** indica que una viga es de acero de 30 cm y una pared es de ladrillo de 15 cm, **antes** de construir. Si el albanil coloca madera donde va acero, el plano (o el supervisor) lo detecta. **Typing** es exactamente eso: un plano que anuncia que tipo de datos espera cada funcion o atributo, y herramientas como `mypy` verifican que el codigo cumpla el plano **sin necesidad de ejecutarlo**.

> Dataclasses reducen la **cantidad** de codigo mecanico; typing aumentan la **claridad** y **seguridad** del codigo. Usados juntos forman la base del codigo Python moderno, limpio y mantenible.

## 1. ¿Por que dataclasses?

En ciencia de datos y desarrollo, la mayoria de modelos son **clases de solo-datos**: contenedores limpios que agrupan campos relacionados (una fila de venta, un registro de cliente, una configuracion). Sin `dataclasses`, escribir cada una de esas clases es repetitivo:

```python
class Usuario:
    def __init__(self, nombre: str, email: str, edad: int):
        self.nombre = nombre
        self.email = email
        self.edad = edad

    def __repr__(self):
        return f"Usuario(nombre={self.nombre!r}, email={self.email!r}, edad={self.edad!r})"

    def __eq__(self, other):
        if not isinstance(other, Usuario):
            return NotImplemented
        return (self.nombre, self.email, self.edad) == (other.nombre, other.email, other.edad)
```

Nota como el **94% del codigo es boilerplate**: Python ya sabe que `nombre`, `email` y `edad` se guardan tal cual, pero igual tienes que escribir `self.x = x` tres veces, ademas de `__repr__` y `__eq__`.

| Aspecto | Clase manual | @dataclass |
|---|---|---|
| **__init__** | Manual (`self.x = x`) | Automatico |
| **__repr__** | Manual (formato a mano) | Automatico y legible |
| **__eq__** | Manual (comparar tuplas) | Automatico |
| **Campos opcionales** | Verboso | Mandas clave `field(default=...)` |
| **Lines de codigo** | ~12 por clase | ~4 por clase |

Los datos (datos de ventas, perfiles, metricas) ganan claridad, y el codigo queda mas corto, mas legible y menos propenso a errores tipograficos.

## 2. @dataclass basico

El decorador `@dataclass` transforma una clase normal en una **clase de datos**: a partir de las anotaciones de tipo de los atributos, genera automaticamente `__init__`, `__repr__`, y `__eq__` (entre otros).

In [ ]:
from dataclasses import dataclass

@dataclass
class Producto:
    sku: str
    nombre: str
    precio: float
    stock: int = 0

# __init__ generado automaticamente
p = Producto("A-001", "Laptop", 8999.99)
print(p)                 # __repr__ generado
print(p == Producto("A-001", "Laptop", 8999.99))  # __eq__ generado
print(f"Stock por defecto: {p.stock}")

### `field()` y `default_factory`

Un campo con valor por defecto **mutable** (lista, dict, set) NO puede usar directamente `= []`, porque el default se compartiria entre todas las instancias (el clasico anti-patron del default mutable). En su lugar se usa `field(default_factory=...)`, que ejecuta la funcion (fabrica) **por cada instancia**, generando objetos independientes.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Pedido:
    id: str
    items: list[str] = field(default_factory=list)      # nueva lista por instancia
    etiquetas: set[str] = field(default_factory=set)   # nuevo set por instancia
    metadatos: dict = field(default_factory=dict)      # nuevo dict por instancia

a = Pedido("P1")
b = Pedido("P2")
a.items.append("Laptop")

print("items de a:", a.items)
print("items de b (INDEPENDIENTE):", b.items)
print("etiquetas:", a.etiquetas)
print("metadatos:", a.metadatos)

### `frozen=True` (inmutabilidad)

Hace que la instancia sea **inmutable** (como una tupla): no se pueden reasignar atributos despues de la creacion. Ideal para valores de configuracion o datos que no deben cambiar. Ademas hace la instancia *hashable* si todos sus campos son hashables, lo que permite usarla en sets y como clave de diccionarios.

In [ ]:
import dataclasses

@dataclass(frozen=True)
class Configuracion:
    esquema: str
    host: str
    puerto: int

cfg = Configuracion("postgresql", "localhost", 5432)

try:
    cfg.puerto = 9999
except dataclasses.FrozenInstanceError as e:
    print("No se puede modificar una instancia frozen:", e)

d = {cfg: "conexion principal"}  # hashable, sirve como clave
print(d[cfg])

### `slots=True` (Python 3.10+)

Genera la clase usando `__slots__`, lo que reduce el **consumo de memoria** y acelera el acceso a atributos. Especialmente util cuando creas millones de objetos (por ejemplo, al cargar datos de un CSV).

In [ ]:
@dataclass(slots=True)
class Fila:
    id: int
    valor: float
    categoria: str

f = Fila(1, 3.14, "A")
print(f)

# Sin slots se reserva un dict por instancia; con slots no existe __dict__
print("¿Tiene __dict__?", hasattr(f, "__dict__"))
print("__slots__:", Fila.__slots__)

### `order=True` y `kw_only=True`

- **`order=True`**: genera los metodos de comparacion (`<`, `<=`, `>`, `>=`) comparando los campos en orden de declaracion. Ut para ordenar listas de objetos.
- **`kw_only=True`** (3.10+): fuerza a que TODOS los campos se pasen por nombre (keyword) en el `__init__`, evitando errores de posicion cuando hay muchos campos.

In [ ]:
@dataclass(order=True)
class Punto:
    x: float
    y: float

pts = [Punto(3, 1), Punto(1, 5), Punto(2, 2)]
print("Ordenados:", sorted(pts))
print("Punto(1,1) < Punto(2,0):", Punto(1, 1) < Punto(2, 0))

In [ ]:
@dataclass(kw_only=True)
class Registro:
    nombre: str
    apellido: str
    dni: str

# Solo se puede construir por keywords:
r = Registro(nombre="Ana", apellido="Lopez", dni="12345678A")
print(r)

try:
    r2 = Registro("Ana", "Lopez", "12345678A")  # posicional -> error
except TypeError as e:
    print("Error por usar posicional:", e)

## 3. __post_init__ y validacion

A veces necesitas hacer algo al construir: **calcular atributos derivados** (que dependen de otros) o **validar** que los datos cumplen reglas de negocio. `@dataclass` te da el enganche `__post_init__`, que se ejecuta automaticamente **despues** de que `__init__` generado asigne todos los campos.

In [ ]:
@dataclass
class Factura:
    base: float
    iva: float = 0.21
    total: float = field(init=False, repr=False)  # NO se pasa por __init__

    def __post_init__(self) -> None:
        if self.base < 0:
            raise ValueError(f"La base no puede ser negativa: {self.base}")
        # Atributo derivado calculado automaticamente
        self.total = self.base * (1 + self.iva)

f = Factura(1000)
print(f"Total con IVA: {f.total:.2f}")

try:
    Factura(-50)
except ValueError as e:
    print("Validacion paso:", e)

Tambien puedes validar en `__post_init__` usando la funcion auxiliar `dataclasses.field(init=False)` combinada con validaciones mas robustas, o simplemente `raise` con mensajes claros para fallar temprano antes de que el mal dato se propague por el sistema.

In [ ]:
@dataclass
class CuentaBancaria:
    titular: str
    saldo: float

    def __post_init__(self) -> None:
        if not self.titular.strip():
            raise ValueError("El titular no puede estar vacio")
        if self.saldo < 0:
            raise ValueError("El saldo no puede ser negativo")

for datos in [("", 100), ("Maria", -5)]:
    try:
        CuentaBancaria(*datos)
    except ValueError as e:
        print("Rechazado:", e)

cb = CuentaBancaria("Maria", 2500)
print("Cuenta valida:", cb)

## 4. Observabilidad: replace, asdict y fields

Las dataclasses vienen con utilidades para inspeccionar y transformar tus objetos sin tocar sus atributos internos directamente.

In [ ]:
from dataclasses import dataclass, asdict, fields, replace

@dataclass
class Cliente:
    id: int
    nombre: str
    email: str
    activo: bool = True

c = Cliente(1, "Luis", "luis@mail.com")

# asdict: convierte la dataclass (y las anidadas) en un dict
print("asdict:", asdict(c))

# fields: lista de metadatos de los campos (nombre, tipo, default...)
print("Campos:", [f.name for f in fields(c)])
print("Tipo de 'id':", fields(c)[0].type)

# replace: crea una COPIA con algunos campos cambiados (inmutable-safe)
c_desactivado = replace(c, activo=False)
print("Original:", c)
print("Copia modificada:", c_desactivado)

| Funcion | Que hace | Ejemplo |
|---|---|---|
| `dataclasses.replace` | Devuelve una copia con campos puntuales cambiados | `replace(c, activo=False)` |
| `dataclasses.asdict` | Convierte (recursivamente) a diccionario | `asdict(c)` -> `{'id': 1, ...}` |
| `dataclasses.astuple` | Convierte a tupla | `astuple(c)` -> `(1, ...)` |
| `dataclasses.fields` | Metadatos del campo (nombre, tipo, default) | `fields(c)[0].name` |
| `dataclasses.is_dataclass` | Comprueba si un objeto es una dataclass | `is_dataclass(c)` |

## 5. Herencia con dataclasses

Las dataclasses soportan herencia: una subclase hereda los campos de la clase base y agrega los suyos. Esto permite construir jerarquias de objetos de datos, como una entidad generica `Empleado` y una especializada `Desarrollador`.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Empleado:
    nombre: str
    salario: float
    departamento: str

@dataclass
class Desarrollador(Empleado):
    lenguajes: list[str] = field(default_factory=list)
    senior: bool = False

dev = Desarrollador("Ana", 45000, "Ingenieria", ["Python", "SQL"], True)
print(dev)
print(f"{dev.nombre} gana {dev.salario} en {dev.departamento}")
print("Lenguajes:", dev.lenguajes)

**⚠️ Orden de los campos en herencia:** los campos de la clase base SIEMPRE se declaran primero. Si un campo de la subclase tiene un valor por defecto, no puede haber campos con default en la clase base despues... En la practica, coloca todos los campos con `default_factory`/`default` al final de la subclase para evitar conflictos de defaults.

In [ ]:
# Errores comunes: campo sin default despues de uno con default

try:
    @dataclass
    class ErrorClase:
        a: int = 0
        b: int  # campo obligatorio DESPUES de uno con default
except TypeError as e:
    print("Tipo de error:", e)
    print("Solucion: ordenar campos, los obligatorios primero:")

@dataclass
class Correcta:
    b: int          # obligatorios primero
    a: int = 0      # con default al final

print(Correcta(7))

## 6. Introduccion a typing

El modulo `typing` (estandar desde Python 3.5, y mucho mas potente en 3.10+) permite anotar tipos con precision. Esas anotaciones no cambian el comportamiento en tiempo de ejecucion (Python sigue siendo dinamico), pero documentan el codigo y permiten que `mypy` u otros linters hagan **type checking estatico**.

### Tipos modernos vs. sintaxis antigua (PEP 604 / PEP 585)

Desde Python 3.10 puedes usar `X | None` en lugar de `Optional[X]`, y `list[int]` en lugar de `List[int]`.

In [ ]:
from __future__ import annotations  # permite anotaciones perezosas (opcional)
from typing import Optional, Union, TypeVar, Generic, Literal, NewType, Annotated, Callable, Any

# Optional[X]  ==  X | None
def f1(x: Optional[int]) -> int | None:
    return x

# Union[X, Y]  ==  X | Y
def f2(v: Union[int, str]) -> int | str:
    return v

# Lista/any de colecciones (PEP 585)
def sumar(numeros: list[int]) -> int:
    return sum(numeros)

def primer_valor(d: dict[str, float]) -> float | None:
    return next(iter(d.values()), None)

print(sumar([1, 2, 3]))
print(f2(10), f2("hola"))
print(primer_valor({"a": 1.5}))

In [ ]:
# Literal: restringe a valores especificos
from typing import Literal

def estado_del_pedido(nuevo_estado: Literal["abierto", "pagado", "enviado", "cerrado"]) -> str:
    return f"Estado actualizado a: {nuevo_estado}"

print(estado_del_pedido("pagado"))
# estado_del_pedido("cancelado")  # mypy lo marcaria como error

# NewType: crea un tipo nominal derivado
from typing import NewType

UsuarioId = NewType("UsuarioId", int)
def get_usuario(uid: UsuarioId) -> str:
    return f"Usuario {uid}"

u = UsuarioId(42)
print(get_usuario(u))

# Annotated: adjunta metadatos adicionales al tipo
from typing import Annotated

Porcentaje = Annotated[float, "valor entre 0 y 100"]
def aplicar_porcentaje(p: Porcentaje) -> float:
    return p / 100

print(aplicar_porcentaje(50.0))

# Callable: tipo de una funcion
from typing import Callable

def aplicar(operacion: Callable[[int, int], int], a: int, b: int) -> int:
    return operacion(a, b)

print(aplicar(lambda x, y: x + y, 3, 4))
print(aplicar(lambda x, y: x * y, 3, 4))

In [ ]:
# TypeVar + Generic[T]: clases y funciones genericas reutilizables
from typing import TypeVar, Generic

T = TypeVar("T")  # variable de tipo

@dataclass
class Caja(Generic[T]):
    contenido: T | None = None

    def poner(self, valor: T) -> None:
        self.contenido = valor

    def sacar(self) -> T | None:
        valor = self.contenido
        self.contenido = None
        return valor

caja_int = Caja[int]()
caja_int.poner(100)
print("Caja int:", caja_int.sacar())

caja_str = Caja[str]()
caja_str.poner("hola")
print("Caja str:", caja_str.sacar())

# Funcion generica con TypeVar y union restringida
def primera(secuencia: list[T]) -> T | None:
    return secuencia[0] if secuencia else None

print(primera([10, 20, 30]))
print(primera(["a", "b"]))

## 7. Protocol (typing.Protocol)

Python utiliza **duck typing**: "si camina como pato y grazna como pato, entonces es un pato". `Protocol` (introducido en **PEP 544**, Python 3.8+) lleva el duck typing al **tipado estatico** permitiendo **structural typing**: defines que una clase **necesita ciertos metodos/atributos**, y cualquier objeto que los tenga satisface el protocolo, sin necesidad de heredar de una clase base.

Comparado con las interfaces estaticas de otros lenguajes:

| Concepto | Interfaces (Java/C#) | Protocol (Python) |
|---|---|---|
| Vinculacion | Implicita (heredar interface) | Implicita (solo tener los metodos) |
| Herencia requerida | Si | No (duck typing) |
| Verificacion | En compilacion | En tiempo de ejecucion con `isinstance` + `@runtime_checkable` |
| Flexibilidad | Baja | Alta (compone clases no relacionadas) |

In [ ]:
from typing import Protocol

class Reproducible(Protocol):
    """Cualquier objeto que pueda reproducir su identificador."""
    def reproducir(self) -> str: ...

@dataclass
class Musica:
    titulo: str
    def reproducir(self) -> str:
        return f"Reproduciendo cancion: {self.titulo}"

@dataclass
class Video:
    nombre: str
    def reproducir(self) -> str:
        return f"Reproduciendo video: {self.nombre}"

# Ambas clases satisfacen Reproducible SIN heredar de nada
def tocar(algo: Reproducible) -> None:
    print(algo.reproducir())

tocar(Musica("Bohemian Rhapsody"))
tocar(Video("Tutorial"))

In [ ]:
# @runtime_checkable permite usar isinstance/isinstance con un Protocol
from typing import Protocol, runtime_checkable

@runtime_checkable
class ConIdentificador(Protocol):
    id: int

@dataclass
class Cliente:
    id: int
    nombre: str

@dataclass
class Pedido:
    id: int
    total: float

# isinstance verifica la presencia del atributo 'id' en runtime
print("Cliente es ConIdentificador:", isinstance(Cliente(1, "Luis"), ConIdentificador))
print("Pedido es ConIdentificador:", isinstance(Pedido(2, 99.9), ConIdentificador))
print("String es ConIdentificador:", isinstance("hola", ConIdentificador))

### Diagrama ASCII: Generic / Protocol / TypeVar

```
                    +----------------------------------------------+
                    |            SISTEMA DE TIPADO PYTHON          |
                    +----------------------------------------------+
                    |                                              |
                    |   TypeVar  ---- define variables de tipo --->   Generic[T]
                    |     |                                          (clases reutilizables)
                    |     |   ejemplo:                                   |
                    |     |   T = TypeVar("T")                          v
                    |     +-----------------------------------> Caja[T] (list, dict, etc.)
                    |
                    |   Protocol ---- define CONTRATOS estructurales
                    |     |        (mypy: duck typing estatico)
                    |     |              |
                    |     |              v
                    |     |        ClaseA ---- cumple --->  (solo importan los metodos)
                    |     |        ClaseB ---- cumple --->
                    |
                    |   Nominal typing:  isinstance / herencia (duck typing dinamico)
                    +----------------------------------------------+

   TypeVar    ->  parametriza con un tipo variable (T, K, V)
   Generic[T] ->  hace una clase reutilizable para cualquier tipo
   Protocol   ->  exige una ESTRUCTURA (metodos), no una clase base
```

## 8. mypy (introduccion)

`mypy` es **type checker estatico** para Python: analiza tu codigo **sin ejecutarlo** y detecta errores de tipos antes de que se manifiesten en produccion. Es opcional instalarlo en el curso, pero el notebook esta preparado para usarlo:

```bash
pip install mypy
mypy archivo.py
```

**Ventajas del type checking estatico:**

- **Detecta errores temprano**: pasa un `None` donde se esperaba un `int` o compara tipos incompatibles.
- **Mejora el autocompletado** en editores (IDE lo usa para sugerencias).
- **Documentacion viva**: el contrato de cada funcion queda explicito en firmas.
- **Refactoring seguro**: cambiar un tipo propaga la verificacion a todos los callers.
- **Eleva la calidad**: encaja en pipelines de CI como una prueba mas.

In [ ]:
# Ejemplo de lo que mypy detectaria (aunque Python lo ejecuta)

## Codigo con error de tipo (mypy lo senialaria):
# def duplicar(x: int) -> int:
#     return x * 2
#
# duplicar("texto")   # mypy: error "Argument 1 has incompatible type str; expected int"

## Codigo correctamente tipado:
def duplicar(x: int) -> int:
    return x * 2

print("duplicar(21) =", duplicar(21))

# Nota: mypy se ejecuta por linea de comandos, no dentro del notebook.
# Para verlo en accion ejecuta en terminal:
#   mypy este_notebook_convertido_a_py.py
print("Ejemplo didactico: mypy corre fuera de Jupyter.")

## Tabla de referencia: constructs de typing

| Construct | Sintaxis (moderna) | Que representa | Ejemplo |
|---|---|---|---|
| `Optional` | `X \| None` | X o ausencia de valor | `def f() -> int \| None` |
| `Union` | `X \| Y` | X o Y | `def f(v: int \| str)` |
| `list[int]` | `list[int]` | lista de enteros | `def f(xs: list[int])` |
| `dict[str, float]` | `dict[str, float]` | mapeo str -> float | `def f(d: dict[str, float])` |
| `TypeVar` | `T = TypeVar("T")` | variable de tipo generica | `def f(xs: list[T]) -> T` |
| `Generic[T]` | `class Caja(Generic[T])` | clase reutilizable por tipo | `Caja[int]` |
| `Literal` | `Literal["a", "b"]` | un valor literal especifico | `estado: Literal["on"]` |
| `NewType` | `UserId = NewType(..., int)` | tipo nominal derivado | `def f(uid: UserId)` |
| `Annotated` | `Annotated[float, "nota"]` | tipo + metadatos | `Annotated[float, "0..100"]` |
| `Callable` | `Callable[[int, int], int]` | firma de funcion | `Callable[[int,int], int]` |
| `Protocol` | `class P(Protocol)` | contrato estructural con metodos | `class Reproducible(Protocol)` |
| `Any` | `Any` | cualquier tipo (escapar verificacion) | `x: Any = "algo"` |
| `Iterator`/`Iterable` | `Iterable[int]` | objeto iterable de enteros | `def f(it: Iterable[int])` |

## Ejercicios

### Ejercicio 1 (guiado): Libro con dataclass

Crea una dataclass `Libro` con `titulo`, `autor`, `anio` y `paginas` (por defecto 0). Usa `__post_init__` para rechazar libros con anio futuro (> 2026) lanzando `ValueError`. Instancia un par de libros y muéstralos.

In [ ]:
from dataclasses import dataclass

@dataclass
class Libro:
    titulo: str
    autor: str
    anio: int
    paginas: int = 0

    def __post_init__(self) -> None:
        if self.anio > 2026:
            raise ValueError(f"El anio {self.anio} es futuro")

b1 = Libro("Clean Code", "Robert Martin", 2008, 464)
b2 = Libro("Fluent Python", "Luciano Ramalho", 2015)
print(b1)
print(b2)

try:
    Libro("Del futuro", "Autor X", 2100)
except ValueError as e:
    print("Rechazado:", e)

### Ejercicio 2 (guiado): Repositorio generico

Usa `Generic[T]` para construir una `PilaGenerica[T]` con `apilar(x: T)`, `desapilar() -> T | None` y `esta_vacia() -> bool`. Pruebala con enteros y con strings.

In [ ]:
from dataclasses import dataclass, field
from typing import Generic, TypeVar

T = TypeVar("T")

@dataclass
class PilaGenerica(Generic[T]):
    _items: list[T] = field(default_factory=list)

    def apilar(self, x: T) -> None:
        self._items.append(x)

    def desapilar(self) -> T | None:
        return self._items.pop() if self._items else None

    def esta_vacia(self) -> bool:
        return not self._items

pila_int: PilaGenerica[int] = PilaGenerica()
pila_int.apilar(1)
pila_int.apilar(2)
print("Desapilo int:", pila_int.desapilar())
print("Vacia?", pila_int.esta_vacia())

pila_str: PilaGenerica[str] = PilaGenerica()
pila_str.apilar("hola")
print("Desapilo str:", pila_str.desapilar())

### Ejercicio 3 (guiado): Protocol para repositorios

Define un `Protocol` `Almacen` con los metodos `guardar(id: str, dato: str) -> None` y `obtener(id: str) -> str | None`. Implementa dos clases no relacionadas (`AlmacenMemoria` y `AlmacenArchivo`) que lo cumplan y usa una funcion generica que acepte cualquier `Almacen`.

In [ ]:
from typing import Protocol, runtime_checkable
from pathlib import Path
from tempfile import TemporaryDirectory

@runtime_checkable
class Almacen(Protocol):
    def guardar(self, id: str, dato: str) -> None: ...
    def obtener(self, id: str) -> str | None: ...

class AlmacenMemoria:
    def __init__(self) -> None:
        self._datos: dict[str, str] = {}
    def guardar(self, id: str, dato: str) -> None:
        self._datos[id] = dato
    def obtener(self, id: str) -> str | None:
        return self._datos.get(id)

class AlmacenArchivo:
    def __init__(self, directorio: str) -> None:
        self._dir = Path(directorio)
        self._dir.mkdir(parents=True, exist_ok=True)
    def guardar(self, id: str, dato: str) -> None:
        (self._dir / f"{id}.txt").write_text(dato, encoding="utf-8")
    def obtener(self, id: str) -> str | None:
        f = self._dir / f"{id}.txt"
        return None if not f.exists() else f.read_text(encoding="utf-8")

def escribir_y_leer(almacen: Almacen, id: str, dato: str) -> None:
    almacen.guardar(id, dato)
    print("Leido:", almacen.obtener(id))

escribir_y_leer(AlmacenMemoria(), "k1", "valor en memoria")

with TemporaryDirectory() as tmp:
    escribir_y_leer(AlmacenArchivo(tmp), "k2", "valor en archivo")

print("AlmacenMemoria cumple Almacen:", isinstance(AlmacenMemoria(), Almacen))
print("AlmacenArchivo cumple Almacen:", isinstance(AlmacenArchivo("x"), Almacen))

### Ejercicio 4 (independiente): Modelo de dominio con dataclasses + Protocol

Construye un mini modelo de dominio de una **tienda online** cumpliendo lo siguiente:

1. `@dataclass` `Producto` con `sku`, `nombre`, `precio` y `stock` (default 0). Valida en `__post_init__` que `precio >= 0`.
2. `@dataclass` `Carrito` con un campo `items: list[Producto]` usando `default_factory`.
3. Metodo `total(carrito: Carrito) -> float` que sume los precios (puede ser una funcion o metodo).
4. Un `Protocol` `Descontable` con metodo `aplicar_descuento(precio: float) -> float`, y una clase `CuponFijo` y una `CuponPorcentual` que lo cumplan.
5. Muestra un ejemplo completo: crea productos, agregalos al carrito, calcula el total y aplica un descuento porcentual.

**Pista:** combina `@dataclass`, `field(default_factory=list)`, `Protocol` y `__post_init__` en una sola solucion.

In [ ]:
from dataclasses import dataclass, field
from typing import Protocol

@dataclass
class Producto:
    sku: str
    nombre: str
    precio: float
    stock: int = 0

    def __post_init__(self) -> None:
        if self.precio < 0:
            raise ValueError(f"El precio no puede ser negativo: {self.precio}")

@dataclass
class Carrito:
    items: list[Producto] = field(default_factory=list)

    def agregar(self, p: Producto) -> None:
        self.items.append(p)

    def total(self) -> float:
        return sum(p.precio for p in self.items)

class Descontable(Protocol):
    def aplicar_descuento(self, precio: float) -> float: ...

@dataclass
class CuponFijo:
    cantidad: float
    def aplicar_descuento(self, precio: float) -> float:
        return max(0.0, precio - self.cantidad)

@dataclass
class CuponPorcentual:
    porcentaje: float  # ej. 20 = 20%
    def aplicar_descuento(self, precio: float) -> float:
        return precio * (1 - self.porcentaje / 100)

# Ejemplo completo
carrito = Carrito()
carrito.agregar(Producto("A1", "Laptop", 8999.99, 5))
carrito.agregar(Producto("A2", "Mouse", 249.50, 20))
carrito.agregar(Producto("A3", "Teclado", 699.00, 15))

total = carrito.total()
print(f"Total sin descuento: {total:.2f}")

descuento: Descontable = CuponPorcentual(15)
final = descuento.aplicar_descuento(total)
print(f"Total con descuento del 15%: {final:.2f}")
print(f"Ahorro: {total - final:.2f}")

## Resumen

En esta sesion aprendiste como **Python moderno** te permite escribir clases de solo-datos y codigo tipado de forma limpia y segura:

- **`@dataclass`** elimina el boilerplate: genera `__init__`, `__repr__` y `__eq__` a partir de las anotaciones.
- **`field(default_factory=...)`** resuelve el problema del default mutable (lista/dict/set independientes por instancia).
- **`frozen=True`** da inmutabilidad y hashabilidad; **`slots=True`** ahorra memoria; **`order=True`** aporta comparaciones; **`kw_only=True`** fuerza keyword args.
- **`__post_init__`** permite validar y calcular atributos derivados al construir la instancia.
- **`replace`, `asdict`, `fields`** dan observabilidad y transformaciones seguras.
- Las dataclasses **heredan** y componen jerarquias de objetos de datos.
- **`typing`** moderno (`X | None`, `list[int]`, `Generic[T]`, `Literal`, `NewType`, `Annotated`, `Callable`, `Protocol`) documenta y verifica el codigo.
- **`Protocol`** lleva el duck typing al tipado estatico (structural typing) sin requerir herencia.
- **`mypy`** hace type checking estatico y detecta errores antes de ejecutar, elevando la calidad del codigo en CI.

> Combina `dataclasses` para la **estructura de datos** y `typing`/`Protocol` para el **contrato** de esa estructura: son la base del codigo Python profesional, limpio y mantenible.